In [ ]:
import sqlite3

# Crear la base de datos
conn = sqlite3.connect('cervezas.db')
cursor = conn.cursor()

# Crear tablas
cursor.executescript('''
CREATE TABLE CERVEZAS (CodC TEXT PRIMARY KEY, Envase TEXT, Capacidad REAL, Stock INTEGER);
CREATE TABLE BARES (CodB TEXT PRIMARY KEY, Cif TEXT, Nombre TEXT, Localidad TEXT);
CREATE TABLE EMPLEADOS (CodE INTEGER PRIMARY KEY, Nombre TEXT, Sueldo REAL);
CREATE TABLE REPARTO (CodE INTEGER, CodB TEXT, CodC TEXT, Fecha TEXT, Cantidad INTEGER,
    FOREIGN KEY(CodE) REFERENCES EMPLEADOS(CodE),
    FOREIGN KEY(CodB) REFERENCES BARES(CodB),
    FOREIGN KEY(CodC) REFERENCES CERVEZAS(CodC));
''')

# Inserción de datos
cervezas_data = [('01','Botella',0.2,3600), ('02','Botella',0.33,1200), ('03','Lata',0.33,2400), ('04','Botella',1,288), ('05','Barril',60,30)]
bares_data = [('001','11111111X','Stop','Villa Botijo'), ('002','22222222Y','Las Vegas','Villa Botijo'), ('003','-','Club Social','Las Ranas'), ('004','33333333Z','Otra Ronda','La Esponja')]
empleados_data = [(1,'Prudencio Caminero',120000), (2,'Vicente Merario',110000), (3,'Valentin Siempre',100000)]
reparto_data = [
    (1,'001','01','2005-10-21',240), (1,'001','02','2005-10-21',48), (1,'002','03','2005-10-22',60),
    (1,'004','05','2005-10-22',4), (2,'002','03','2005-10-22',48), (2,'002','05','2005-10-23',2),
    (2,'004','01','2005-10-23',480), (2,'004','02','2005-10-24',72), (3,'003','03','2005-10-24',48), (3,'003','04','2005-10-25',20)
]

cursor.executemany('INSERT INTO CERVEZAS VALUES (?,?,?,?)', cervezas_data)
cursor.executemany('INSERT INTO BARES VALUES (?,?,?,?)', bares_data)
cursor.executemany('INSERT INTO EMPLEADOS VALUES (?,?,?)', empleados_data)
cursor.executemany('INSERT INTO REPARTO VALUES (?,?,?,?,?)', reparto_data)
conexion.commit()


## Ejercicios
---
### 1. Nombre de los empleados que repartieron al bar Stop (17-23 oct 2005)

In [10]:
cursor.execute("""
    SELECT DISTINCT E.Nombre 
    FROM EMPLEADOS E 
    JOIN REPARTO R ON E.CodE = R.CodE 
    JOIN BARES B ON R.CodB = B.CodB
    WHERE B.Nombre = 'Stop' 
      AND R.Fecha BETWEEN '2005-10-17' AND '2005-10-23';
""")
for fila in cursor.fetchall():
    print(fila[0])




Prudencio Caminero


### 2. Cif y nombre de los bares con Botella < 1L, por localidad

In [11]:
# 2. Obtener el Cif y nombre de los bares...
cursor.execute("""
    SELECT DISTINCT B.Cif, B.Nombre 
    FROM BARES B
    JOIN REPARTO R ON B.CodB = R.CodB 
    JOIN CERVEZAS C ON R.CodC = C.CodC
    WHERE C.Envase = 'Botella' AND C.Capacidad < 1 
    ORDER BY B.Localidad;
""")
for fila in cursor.fetchall():
    print(f"CIF: {fila[0]} | Nombre: {fila[1]}")


CIF: 33333333Z | Nombre: Otra Ronda
CIF: 11111111X | Nombre: Stop


### 3. Repartos de Prudencio Caminero

In [12]:
# 3. Repartos realizados por Prudencio Caminero
cursor.execute("""
    SELECT B.Nombre, C.Envase, C.Capacidad, R.Fecha, R.Cantidad 
    FROM REPARTO R 
    JOIN EMPLEADOS E ON R.CodE = E.CodE
    JOIN BARES B ON R.CodB = B.CodB 
    JOIN CERVEZAS C ON R.CodC = C.CodC
    WHERE E.Nombre = 'Prudencio Caminero';
""")
for fila in cursor.fetchall():
    print(fila)

('Stop', 'Botella', 0.2, '2005-10-21', 240)
('Stop', 'Botella', 0.33, '2005-10-21', 48)
('Las Vegas', 'Lata', 0.33, '2005-10-22', 60)
('Otra Ronda', 'Barril', 60.0, '2005-10-22', 4)


### 4. Bares con envase botella y capacidad 0.2 o 0.33

In [13]:
# 4. Obtener los bares (botella y capacidad 0.2 ó 0.33)
cursor.execute("""
    SELECT DISTINCT B.Nombre 
    FROM BARES B
    JOIN REPARTO R ON B.CodB = R.CodB 
    JOIN CERVEZAS C ON R.CodC = C.CodC
    WHERE C.Envase = 'Botella' AND (C.Capacidad = 0.2 OR C.Capacidad = 0.33);
""")
for fila in cursor.fetchall():
    print(fila[0])

Stop
Otra Ronda


### 5. Empleados que repartieron a 'Stop' y 'Las Vegas' (Botellas)

In [ ]:
# Empleados en "Stop" Y "Las Vegas" (solo botellas)
cursor.execute("""
    SELECT E.Nombre FROM EMPLEADOS E
    JOIN REPARTO R ON E.CodE = R.CodE 
    JOIN BARES B ON R.CodB = B.CodB 
    JOIN CERVEZAS C ON R.CodC = C.CodC
    WHERE B.Nombre = 'Stop' AND C.Envase = 'Botella'
    INTERSECT
    SELECT E.Nombre FROM EMPLEADOS E
    JOIN REPARTO R ON E.CodE = R.CodE 
    JOIN BARES B ON R.CodB = B.CodB 
    JOIN CERVEZAS C ON R.CodC = C.CodC
    WHERE B.Nombre = 'Las Vegas' AND C.Envase = 'Botella';
""")

resultados5 = cursor.fetchall()

if len(resultados5) > 0:
    print("✅ Empleados que cumplen la condición:")
    for fila in resultados5:
        print(f"- {fila[0]}")
else:
    print("No se encontraron empleados que hayan repartido 'Botellas' en AMBOS bares (Stop y Las Vegas).")


No se encontraron empleados que hayan repartido 'Botellas' en AMBOS bares (Stop y Las Vegas).


### 6. Viajes fuera de Villa Botijo

In [ ]:
#Viajes fuera de Villa Botijo

cursor.execute("""
    SELECT E.Nombre, COUNT(*) 
    FROM EMPLEADOS E
    JOIN REPARTO R ON E.CodE = R.CodE 
    JOIN BARES B ON R.CodB = B.CodB
    WHERE B.Localidad != 'Villa Botijo' 
    GROUP BY E.Nombre;
""")
for fila in cursor.fetchall():
    print(f"Empleado: {fila[0]} | Viajes: {fila[1]}")

Empleado: Prudencio Caminero | Viajes: 1
Empleado: Valentin Siempre | Viajes: 2
Empleado: Vicente Merario | Viajes: 2


### 7. Bar que más litros ha comprado

In [16]:
#Bar con más litros comprados

cursor.execute("""
    SELECT B.Nombre, B.Localidad, SUM(R.Cantidad * C.Capacidad) as Total
    FROM BARES B 
    JOIN REPARTO R ON B.CodB = R.CodB 
    JOIN CERVEZAS C ON R.CodC = C.CodC
    GROUP BY B.CodB 
    ORDER BY Total DESC LIMIT 1;
""")
res = cursor.fetchone()
print(f"El bar que más compra es {res[0]} ({res[1]}) con {res[2]} litros.")

El bar que más compra es Otra Ronda (La Esponja) con 359.76 litros.


### 8. Bares con todos los tipos de botella < 1L

In [17]:
#Bares con TODOS los tipos de Botella < 1L
cursor.execute("""
    SELECT B.Nombre FROM BARES B 
    WHERE NOT EXISTS (
        SELECT CodC FROM CERVEZAS WHERE Envase = 'Botella' AND Capacidad < 1
        EXCEPT
        SELECT CodC FROM REPARTO R WHERE R.CodB = B.CodB
    );
""")
for fila in cursor.fetchall():
    print(fila[0])

Stop
Otra Ronda


### 9. Subir sueldo al empleado que más días trabajó

In [ ]:
#Subida de sueldo (UPDATE) / Limit 1 se queda con el que mas, y el distinct filtra por dias diferentes, no cantidad de repartos.
cursor.execute("""
    UPDATE EMPLEADOS SET Sueldo = Sueldo * 1.05 
    WHERE CodE = (
        SELECT CodE FROM REPARTO 
        GROUP BY CodE 
        ORDER BY COUNT(DISTINCT Fecha) DESC LIMIT 1
    );
""")
conn.commit()
print("Sueldo actualizado correctamente.")

Sueldo actualizado correctamente.


### 10. Insertar nuevo reparto

In [19]:
#Insertar nuevo reparto (INSERT)
cursor.execute("""
    INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad)
    VALUES (
        (SELECT CodE FROM EMPLEADOS WHERE Nombre = 'Vicente Merario'),
        (SELECT CodB FROM BARES WHERE Nombre = 'Stop'),
        (SELECT CodC FROM CERVEZAS WHERE Envase = 'Lata' LIMIT 1),
        '2005-10-26', 48
    );
""")
conn.commit()
print("Nuevo reparto insertado con éxito.")

Nuevo reparto insertado con éxito.
